# Imports

In [1]:
import torch
import sys


from torch_geometric.datasets import Planetoid

c:\faculdade\Tabalho-Final-XAI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import random
import torch.nn.functional as F
from torch_geometric.nn import  HANConv
from torch_geometric.datasets import DBLP
import os
import pandas as pd

In [3]:
from itertools import combinations
from tqdm import tqdm
import re
from scipy.stats import spearmanr

In [4]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

In [5]:
import numpy as np
import warnings
import pickle
from pathlib import Path

In [6]:
from torch_geometric.explain import Explainer
from torch_geometric.explain.algorithm import PGExplainer
from torch_geometric.explain.config import ModelConfig

# Dados

In [7]:
def load_dataset(name):

    if name in [
        "Cora",
        "CiteSeer",
        "PubMed"
    ]:

        dataset = Planetoid(
            root=f"data/{name}",
            name=name
        )

        return dataset

    elif name == "DBLP":

        dataset = DBLP(
            root="data/DBLP"
        )

        return dataset

    else:

        raise ValueError(
            f"Dataset {name} não suportado."
        )

DBLP

In [34]:
from torch_geometric.data import HeteroData


In [35]:
DBdataset = DBLP(
    root='data/DBLP'
)

In [36]:
dblp = DBdataset[0]


dblp_new = HeteroData()

# nós
dblp_new['author'].x = dblp['author'].x
dblp_new['author'].y = dblp['author'].y
dblp_new['author'].train_mask = dblp['author'].train_mask
dblp_new['author'].val_mask = dblp['author'].val_mask
dblp_new['author'].test_mask = dblp['author'].test_mask

dblp_new['paper'].x = dblp['paper'].x

dblp_new['term'].x = dblp['term'].x

In [37]:
dblp_new['author', 'to', 'paper'].edge_index = \
    dblp['author', 'to', 'paper'].edge_index

dblp_new['paper', 'to', 'author'].edge_index = \
    dblp['paper', 'to', 'author'].edge_index

dblp_new['paper', 'to', 'term'].edge_index = \
    dblp['paper', 'to', 'term'].edge_index

dblp_new['term', 'to', 'paper'].edge_index = \
    dblp['term', 'to', 'paper'].edge_index

In [38]:
print(dblp_new.metadata())

(['author', 'paper', 'term'], [('author', 'to', 'paper'), ('paper', 'to', 'author'), ('paper', 'to', 'term'), ('term', 'to', 'paper')])


In [39]:
print(dblp_new.node_types)

['author', 'paper', 'term']


In [40]:
print(dblp_new.edge_types)

[('author', 'to', 'paper'), ('paper', 'to', 'author'), ('paper', 'to', 'term'), ('term', 'to', 'paper')]


In [41]:
print(dblp_new["author"])

{'x': tensor([[0., 0., 1.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]]), 'y': tensor([2, 2, 3,  ..., 0, 0, 0]), 'train_mask': tensor([False, False, False,  ..., False, False, False]), 'val_mask': tensor([False, False,  True,  ..., False, False, False]), 'test_mask': tensor([ True,  True, False,  ...,  True,  True,  True])}


In [45]:
for node_type in dblp_new.node_types:

    print()
    print(node_type)
    print(dblp_new[node_type])


author
{'x': tensor([[0., 0., 1.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]]), 'y': tensor([2, 2, 3,  ..., 0, 0, 0]), 'train_mask': tensor([False, False, False,  ..., False, False, False]), 'val_mask': tensor([False, False,  True,  ..., False, False, False]), 'test_mask': tensor([ True,  True, False,  ...,  True,  True,  True])}

paper
{'x': tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])}

term
{'x': tensor([[-0.6924, -0.4659,  1.1540,  ...,  0.9178,  0.1995, -0.6360],
        [ 1.2031, -0.4003,  0.0740,  ...,  1.3262, -0.3325,  0.8198],
        [ 0.3748,  0.5731,  0.4802,  ...,  1.1522,  0.6010,

In [44]:
for node_type in dblp_new.node_types:

    print(node_type)
    print("num_nodes =", dblp_new[node_type].num_nodes)

    if 'x' in dblp_new[node_type]:
        print("x.shape =", dblp_new[node_type].x.shape)

    print()

author
num_nodes = 4057
x.shape = torch.Size([4057, 334])

paper
num_nodes = 14328
x.shape = torch.Size([14328, 4231])

term
num_nodes = 7723
x.shape = torch.Size([7723, 50])



# Modelo

In [24]:
class HAN(torch.nn.Module):

    def __init__(
        self,
        in_channels,
        hidden_channels,
        out_channels,
        metadata,
        heads=8,
        dropout=0.5
    ):
        super().__init__()

        self.conv = HANConv(
            in_channels=in_channels,
            out_channels=hidden_channels,
            metadata=metadata,
            heads=heads
        )

        self.dropout = torch.nn.Dropout(
            p=dropout
        )

        self.lin = torch.nn.Linear(
            hidden_channels,
            out_channels
        )

    def forward(
        self,
        x_dict,
        edge_index_dict
    ):

        x_dict = self.conv(
            x_dict,
            edge_index_dict
        )

        x_author = x_dict[
            'author'
        ]

        x_author = self.dropout(
            x_author
        )

        out = self.lin(
            x_author
        )

        return out

# Treinamento

## Apoio

In [19]:
def train(model,data,optimizer):
    model.train()
    optimizer.zero_grad()

    out = model(data.x_dict, data.edge_index_dict)

    criterion = torch.nn.CrossEntropyLoss()

    loss = criterion(out[data['author'].train_mask],
                      data['author'].y[data['author'].train_mask])

    loss.backward()
    optimizer.step()

    return loss.item()

In [46]:
@torch.no_grad()
def evaluate(model, data):

    model.eval()

    out = model(
        data.x_dict,
        data.edge_index_dict
    )

    pred = out.argmax(dim=1)

    train_acc = (
        pred[data['author'].train_mask]
        ==
        data['author'].y[data['author'].train_mask]
    ).float().mean()

    val_acc = (
        pred[data['author'].val_mask]
        ==
        data['author'].y[data['author'].val_mask]
    ).float().mean()

    test_acc = (
        pred[data['author'].test_mask]
        ==
        data['author'].y[data['author'].test_mask]
    ).float().mean()

    return (
        train_acc.item(),
        val_acc.item(),
        test_acc.item()
    )

## Baseline

In [45]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

data = dblp.to(device)

model = HAN(
    in_channels=-1,  # PyG infere automaticamente
    hidden_channels=64,
    out_channels=4,  # DBLP tem 4 classes de autores
    metadata=metadata,
    heads=8
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=0.001)

In [ ]:
for epoch in range(1, 101):
    loss = train(model,data,optimizer)
    acc = evaluate(model,data)

    print(f"Epoch {epoch:03d}, Loss: {loss:.4f}, Acc: {acc:.4f}")

Epoch 001, Loss: 1.3948, Acc: 0.4105
Epoch 002, Loss: 1.3614, Acc: 0.5570
Epoch 003, Loss: 1.3223, Acc: 0.6033
Epoch 004, Loss: 1.2740, Acc: 0.6257
Epoch 005, Loss: 1.2194, Acc: 0.6371
Epoch 006, Loss: 1.1600, Acc: 0.6472
Epoch 007, Loss: 1.0962, Acc: 0.6641
Epoch 008, Loss: 1.0286, Acc: 0.6779
Epoch 009, Loss: 0.9583, Acc: 0.6905
Epoch 010, Loss: 0.8865, Acc: 0.7006
Epoch 011, Loss: 0.8143, Acc: 0.7080
Epoch 012, Loss: 0.7431, Acc: 0.7169
Epoch 013, Loss: 0.6740, Acc: 0.7277
Epoch 014, Loss: 0.6079, Acc: 0.7353
Epoch 015, Loss: 0.5455, Acc: 0.7452
Epoch 016, Loss: 0.4875, Acc: 0.7538
Epoch 017, Loss: 0.4341, Acc: 0.7651
Epoch 018, Loss: 0.3856, Acc: 0.7753
Epoch 019, Loss: 0.3419, Acc: 0.7826
Epoch 020, Loss: 0.3031, Acc: 0.7885
Epoch 021, Loss: 0.2690, Acc: 0.7967
Epoch 022, Loss: 0.2392, Acc: 0.8017
Epoch 023, Loss: 0.2135, Acc: 0.8038
Epoch 024, Loss: 0.1915, Acc: 0.8081
Epoch 025, Loss: 0.1727, Acc: 0.8084
Epoch 026, Loss: 0.1566, Acc: 0.8075
Epoch 027, Loss: 0.1429, Acc: 0.8066
E

## Variantes

In [25]:
def generate_han_variants():

    variants = []

    for hidden_channels in [32, 64, 128]:

        for heads in [2, 4, 8]:

            for dropout in [0.3, 0.5, 0.7]:

                variants.append({

                    "hidden_channels":
                        hidden_channels,

                    "heads":
                        heads,

                    "dropout":
                        dropout

                })

    return variants

In [48]:
SAVE_DIR = "C:\\faculdade\\Tabalho-Final-XAI\\top_models"

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)

results = []

dataset_name = "DBLP"

print(f"\n{'='*50}")
print(f"Dataset: {dataset_name}")
print(f"{'='*50}")

data = dblp_new

variants = generate_han_variants()

for variant_id, config in enumerate(variants):

    print(
        f"\nVariant {variant_id}"
    )

    model = HAN(
        in_channels={

            node_type:
            data[node_type].num_features

            for node_type in data.node_types

            if 'x' in data[node_type]

        },

        metadata=data.metadata(),

        hidden_channels=config["hidden_channels"],

        out_channels=4,

        heads=config["heads"],

        dropout=config["dropout"]

    ).to(device)

    optimizer = torch.optim.Adam(

        model.parameters(),

        lr=0.005,

        weight_decay=5e-4

    )

    best_val = 0
    best_test = 0

    for epoch in range(1, 201):

        loss = train(
            model,
            data,
            optimizer
        )

        train_acc, val_acc, test_acc = evaluate(
            model,
            data
        )

        if val_acc > best_val:

            best_val = val_acc
            best_test = test_acc

    result = {

        "dataset": dataset_name,

        "model": "HAN",

        "variant_id": variant_id,

        "hidden_channels":
            config["hidden_channels"],

        "heads":
            config["heads"],

        "dropout":
            config["dropout"],

        "best_val":
            best_val,

        "best_test":
            best_test

    }

    results.append(result)

    print(
        f"Variant {variant_id} | "
        f"Val={best_val:.4f} | "
        f"Test={best_test:.4f}"
    )



Dataset: DBLP

Variant 0
Variant 0 | Val=0.7850 | Test=0.8014

Variant 1
Variant 1 | Val=0.7775 | Test=0.8041

Variant 2
Variant 2 | Val=0.7925 | Test=0.8078

Variant 3
Variant 3 | Val=0.7900 | Test=0.8075

Variant 4
Variant 4 | Val=0.7925 | Test=0.8124

Variant 5
Variant 5 | Val=0.7800 | Test=0.8103

Variant 6
Variant 6 | Val=0.7925 | Test=0.8020

Variant 7
Variant 7 | Val=0.7950 | Test=0.8096

Variant 8
Variant 8 | Val=0.7950 | Test=0.8099

Variant 9
Variant 9 | Val=0.7825 | Test=0.8035

Variant 10
Variant 10 | Val=0.7800 | Test=0.8121

Variant 11
Variant 11 | Val=0.7950 | Test=0.8142

Variant 12
Variant 12 | Val=0.7850 | Test=0.8010

Variant 13
Variant 13 | Val=0.7925 | Test=0.8133

Variant 14
Variant 14 | Val=0.7900 | Test=0.8115

Variant 15
Variant 15 | Val=0.7925 | Test=0.8075

Variant 16
Variant 16 | Val=0.7900 | Test=0.8112

Variant 17
Variant 17 | Val=0.7925 | Test=0.8201

Variant 18
Variant 18 | Val=0.7850 | Test=0.8118

Variant 19
Variant 19 | Val=0.7800 | Test=0.8112

Vari

In [53]:
results_df = pd.DataFrame(results)
results_df

,dataset,model,variant_id,hidden_channels,heads,dropout,best_val,best_test
0,DBLP,HAN,0,32,2,0.3,0.7850,0.801351
1,DBLP,HAN,1,32,2,0.5,0.7775,0.804114
2,DBLP,HAN,2,32,2,0.7,0.7925,0.807799
3,DBLP,HAN,3,32,4,0.3,0.7900,0.807492
4,DBLP,HAN,4,32,4,0.5,0.7925,0.812404
5,DBLP,HAN,5,32,4,0.7,0.7800,0.810255
6,DBLP,HAN,6,32,8,0.3,0.7925,0.801965
7,DBLP,HAN,7,32,8,0.5,0.7950,0.809641
8,DBLP,HAN,8,32,8,0.7,0.7950,0.809948
9,DBLP,HAN,9,64,2,0.3,0.7825,0.803500


In [50]:
top_models = (

    results_df

    .sort_values(

        by="best_test",

        ascending=False

    )

    .head(3)

)

In [52]:
for _, row in top_models.iterrows():

    filename = (

        f"DBLP_HAN"

        f"_h{row['hidden_channels']}"

        f"_heads{row['heads']}"

        f"_d{row['dropout']}"

        ".pth"

    )
    
    model = model = HAN(
        in_channels={

            node_type:
            data[node_type].num_features

            for node_type in data.node_types

            if 'x' in data[node_type]

        },

        metadata=data.metadata(),

        hidden_channels=row["hidden_channels"],

        out_channels=4,

        heads=row["heads"],

        dropout=row["dropout"]

    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.01,
        weight_decay=5e-4
    )

    for epoch in range(200):

        train(
            model,
            data,
            optimizer
        )

    torch.save(

        model.state_dict(),

        os.path.join(
            SAVE_DIR,
            filename
        )

    )

    print(
        f"Modelo salvo: {filename}"
    )

Modelo salvo: DBLP_HAN_h64_heads8_d0.7.pth
Modelo salvo: DBLP_HAN_h128_heads8_d0.7.pth
Modelo salvo: DBLP_HAN_h128_heads2_d0.7.pth


# Conformidade

In [ ]:
TOP_DIR = "C:\\faculdade\\Tabalho-Final-XAI\\top_models"

### Dados

In [ ]:
dblp = DBdataset[0]


dblp_new = HeteroData()

# nós
dblp_new['author'].x = dblp['author'].x
dblp_new['author'].y = dblp['author'].y
dblp_new['author'].train_mask = dblp['author'].train_mask
dblp_new['author'].val_mask = dblp['author'].val_mask
dblp_new['author'].test_mask = dblp['author'].test_mask

dblp_new['paper'].x = dblp['paper'].x

dblp_new['term'].x = dblp['term'].x

dblp_new['author', 'to', 'paper'].edge_index = \
    dblp['author', 'to', 'paper'].edge_index

dblp_new['paper', 'to', 'author'].edge_index = \
    dblp['paper', 'to', 'author'].edge_index

dblp_new['paper', 'to', 'term'].edge_index = \
    dblp['paper', 'to', 'term'].edge_index

dblp_new['term', 'to', 'paper'].edge_index = \
    dblp['term', 'to', 'paper'].edge_index

### Apoio

In [ ]:
from itertools import combinations
import pandas as pd

import torch

def agreement_all_models(
    models,
    data,
    node_mask=None
):

    if node_mask is None:
        node_mask = torch.ones(
            data.num_nodes,
            dtype=torch.bool,
            device=data.x.device
        )

    predictions = []

    for model in models.values():

        model.eval()

        with torch.no_grad():

            pred = model(
                data.x,
                data.edge_index
            ).argmax(dim=1)

        predictions.append(pred)

    predictions = torch.stack(predictions)

    agreement = (
        predictions ==
        predictions[0]
    ).all(dim=0)

    agreement = agreement[node_mask]

    percentage = (
        agreement.float().mean().item()
        * 100
    )

    return percentage

In [ ]:
from itertools import combinations
import pandas as pd

def pairwise_agreement(
    models,
    data,
    node_mask=None
):

    if node_mask is None:
        node_mask = torch.ones(
            data.num_nodes,
            dtype=torch.bool,
            device=data.x.device
        )

    predictions = {}

    for model_name, model in models.items():

        model.eval()

        with torch.no_grad():

            pred = model(
                data.x,
                data.edge_index
            ).argmax(dim=1)

        predictions[model_name] = pred

    results = []

    for model_a, model_b in combinations(
        models.keys(),
        2
    ):

        agreement = (

            predictions[model_a][node_mask]
            ==
            predictions[model_b][node_mask]

        ).float().mean().item()

        results.append({

            "Model A": model_a,

            "Model B": model_b,

            "Agreement (%)":
                agreement * 100

        })

    return pd.DataFrame(
        results
    )

### Cora

GCN

In [ ]:
cora_gcn_models = load_saved_models(
    "Cora",
    "GCN"
)

In [ ]:
agreement_GCN_cora = agreement_all_models(
    cora_gcn_models,
    cora,
    node_mask=cora.test_mask
)

print(
    f"Concordância dos 3 modelos: {agreement_GCN_cora:.2f}%"
)

Concordância dos 3 modelos: 92.40%


In [ ]:
pairwise__GCN_cora_df = pairwise_agreement(
    cora_gcn_models,
    cora,
    node_mask=cora.test_mask
)

pairwise__GCN_cora_df

,Model A,Model B,Agreement (%)
0,Cora_GCN_h16_d0.5.pth,Cora_GCN_h64_d0.5.pth,93.500000
1,Cora_GCN_h16_d0.5.pth,Cora_GCN_h64_d0.7.pth,94.800001
2,Cora_GCN_h64_d0.5.pth,Cora_GCN_h64_d0.7.pth,96.200001


GAT

In [ ]:
cora_GAT_models = load_saved_models(
    "Cora",
    "GAT"
)

In [ ]:
agreement_GAT_cora = agreement_all_models(
    cora_GAT_models,
    cora,
    node_mask=cora.test_mask
)

print(
    f"Concordância dos 3 modelos: {agreement_GAT_cora:.2f}%"
)

Concordância dos 3 modelos: 88.40%


In [ ]:
pairwise__GAT_cora_df = pairwise_agreement(
    cora_GAT_models,
    cora,
    node_mask=cora.test_mask
)

pairwise__GAT_cora_df

,Model A,Model B,Agreement (%)
0,Cora_GAT_h32_d0.6_heads4.pth,Cora_GAT_h8_d0.6_heads4.pth,92.600000
1,Cora_GAT_h32_d0.6_heads4.pth,Cora_GAT_h8_d0.6_heads8.pth,92.699999
2,Cora_GAT_h8_d0.6_heads4.pth,Cora_GAT_h8_d0.6_heads8.pth,91.000003


GraphSAGE

In [ ]:
cora_GraphSAGE_models = load_saved_models(
    "Cora",
    "GraphSAGE"
)

In [ ]:
agreement_GraphSAGE_cora = agreement_all_models(
    cora_GraphSAGE_models,
    cora,
    node_mask=cora.test_mask
)

print(
    f"Concordância dos 3 modelos: {agreement_GraphSAGE_cora:.2f}%"
)

Concordância dos 3 modelos: 91.10%


In [ ]:
pairwise__GraphSAGE_cora_df = pairwise_agreement(
    cora_GraphSAGE_models,
    cora,
    node_mask=cora.test_mask
)

pairwise__GraphSAGE_cora_df

,Model A,Model B,Agreement (%)
0,Cora_GraphSAGE_h32_d0.3.pth,Cora_GraphSAGE_h32_d0.7.pth,92.199999
1,Cora_GraphSAGE_h32_d0.3.pth,Cora_GraphSAGE_h64_d0.3.pth,97.200000
2,Cora_GraphSAGE_h32_d0.7.pth,Cora_GraphSAGE_h64_d0.3.pth,92.299998


### CiteSeer

GCN

In [ ]:
citeseer_gcn_models = load_saved_models(
    "CiteSeer",
    "GCN"
)

In [ ]:
agreement_GCN_citeseer = agreement_all_models(
    citeseer_gcn_models,
    citeseer,
    node_mask=citeseer.test_mask
)

print(
    f"Concordância dos 3 modelos: {agreement_GCN_citeseer:.2f}%"
)

Concordância dos 3 modelos: 91.50%


In [ ]:
pairwise__GCN_citeseer_df = pairwise_agreement(
    citeseer_gcn_models,
    citeseer,
    node_mask=citeseer.test_mask
)

pairwise__GCN_citeseer_df

,Model A,Model B,Agreement (%)
0,CiteSeer_GCN_h64_d0.3.pth,CiteSeer_GCN_h64_d0.5.pth,94.499999
1,CiteSeer_GCN_h64_d0.3.pth,CiteSeer_GCN_h64_d0.7.pth,92.600000
2,CiteSeer_GCN_h64_d0.5.pth,CiteSeer_GCN_h64_d0.7.pth,95.700002


GAT

In [ ]:
citeseer_GAT_models = load_saved_models(
    "CiteSeer",
    "GAT"
)

In [ ]:
agreement_GAT_citeseer = agreement_all_models(
    citeseer_GAT_models,
    citeseer,
    node_mask=citeseer.test_mask
)

print(
    f"Concordância dos 3 modelos: {agreement_GAT_citeseer:.2f}%"
)

Concordância dos 3 modelos: 78.20%


In [ ]:
pairwise__GAT_citeseer_df = pairwise_agreement(
    citeseer_GAT_models,
    citeseer,
    node_mask=citeseer.test_mask
)

pairwise__GAT_citeseer_df

,Model A,Model B,Agreement (%)
0,CiteSeer_GAT_h32_d0.4_heads4.pth,CiteSeer_GAT_h32_d0.6_heads8.pth,82.800001
1,CiteSeer_GAT_h32_d0.4_heads4.pth,CiteSeer_GAT_h8_d0.4_heads4.pth,85.799998
2,CiteSeer_GAT_h32_d0.6_heads8.pth,CiteSeer_GAT_h8_d0.4_heads4.pth,86.199999


GraphSAGE

In [ ]:
citeseer_GraphSAGE_models = load_saved_models(
    "CiteSeer",
    "GraphSAGE"
)

In [ ]:
agreement_GraphSAGE_citeseer = agreement_all_models(
    citeseer_GraphSAGE_models,
    citeseer,
    node_mask=citeseer.test_mask
)

print(
    f"Concordância dos 3 modelos: {agreement_GraphSAGE_citeseer:.2f}%"
)

Concordância dos 3 modelos: 91.00%


In [ ]:
pairwise__GraphSAGE_citeseer_df = pairwise_agreement(
    citeseer_GraphSAGE_models,
    citeseer,
    node_mask=citeseer.test_mask
)

pairwise__GraphSAGE_citeseer_df

,Model A,Model B,Agreement (%)
0,CiteSeer_GraphSAGE_h32_d0.3.pth,CiteSeer_GraphSAGE_h64_d0.3.pth,94.700003
1,CiteSeer_GraphSAGE_h32_d0.3.pth,CiteSeer_GraphSAGE_h64_d0.5.pth,92.799997
2,CiteSeer_GraphSAGE_h64_d0.3.pth,CiteSeer_GraphSAGE_h64_d0.5.pth,94.300002


### PubMed

GCN

In [ ]:
pubmed_gcn_models = load_saved_models(
    "PubMed",
    "GCN"
)

In [ ]:
agreement_GCN_pubmed = agreement_all_models(
    pubmed_gcn_models,
    pubmed,
    node_mask=pubmed.test_mask
)

print(
    f"Concordância dos 3 modelos: {agreement_GCN_pubmed:.2f}%"
)

Concordância dos 3 modelos: 95.70%


In [ ]:
pairwise__GCN_pubmed_df = pairwise_agreement(
    pubmed_gcn_models,
    pubmed,
    node_mask=pubmed.test_mask
)

pairwise__GCN_pubmed_df

,Model A,Model B,Agreement (%)
0,PubMed_GCN_h16_d0.7.pth,PubMed_GCN_h32_d0.3.pth,96.799999
1,PubMed_GCN_h16_d0.7.pth,PubMed_GCN_h32_d0.5.pth,96.300000
2,PubMed_GCN_h32_d0.3.pth,PubMed_GCN_h32_d0.5.pth,98.299998


GAT

In [ ]:
pubmed_GAT_models = load_saved_models(
    "PubMed",
    "GAT"
)

In [ ]:
agreement_GAT_pubmed = agreement_all_models(
    pubmed_GAT_models,
    pubmed,
    node_mask=pubmed.test_mask
)

print(
    f"Concordância dos 3 modelos: {agreement_GAT_pubmed:.2f}%"
)

Concordância dos 3 modelos: 91.40%


In [ ]:
pairwise__GAT_pubmed_df = pairwise_agreement(
    pubmed_GAT_models,
    pubmed,
    node_mask=pubmed.test_mask
)

pairwise__GAT_pubmed_df

,Model A,Model B,Agreement (%)
0,PubMed_GAT_h16_d0.4_heads4.pth,PubMed_GAT_h32_d0.6_heads8.pth,93.099999
1,PubMed_GAT_h16_d0.4_heads4.pth,PubMed_GAT_h8_d0.6_heads8.pth,94.499999
2,PubMed_GAT_h32_d0.6_heads8.pth,PubMed_GAT_h8_d0.6_heads8.pth,95.200002


GraphSAGE

In [ ]:
pubmed_GraphSAGE_models = load_saved_models(
    "PubMed",
    "GraphSAGE"
)

In [ ]:
agreement_GraphSAGE_pubmed = agreement_all_models(
    pubmed_GraphSAGE_models,
    pubmed,
    node_mask=pubmed.test_mask
)

print(
    f"Concordância dos 3 modelos: {agreement_GraphSAGE_pubmed:.2f}%"
)

Concordância dos 3 modelos: 91.70%


In [ ]:
pairwise__GraphSAGE_pubmed_df = pairwise_agreement(
    pubmed_GraphSAGE_models,
    pubmed,
    node_mask=pubmed.test_mask
)

pairwise__GraphSAGE_pubmed_df

,Model A,Model B,Agreement (%)
0,PubMed_GraphSAGE_h16_d0.5.pth,PubMed_GraphSAGE_h32_d0.3.pth,96.499997
1,PubMed_GraphSAGE_h16_d0.5.pth,PubMed_GraphSAGE_h64_d0.5.pth,93.199998
2,PubMed_GraphSAGE_h32_d0.3.pth,PubMed_GraphSAGE_h64_d0.5.pth,93.699998
